# Lab 01 — Python On-Ramp
**CE/ARCE 2XX · AI Literacy for Civil and Architectural Engineering** · Week 1 lab · about 75 minutes

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TensorBuildLab/AI-Literacy-CE-ARCE/blob/main/labs/lab01-python-onramp/Lab01_Python_OnRamp.ipynb)

Welcome to your first lab! You do **not** need any coding experience. Today you will learn just enough Python to run real AI models — and then you will run two of them: one that looks at photos of concrete, and one that writes text.

### What you will be able to do
1. Open, run, and save a Google Colab notebook, and tell code cells from text cells.
2. Use Python variables, lists, dictionaries, loops, `if` statements, and functions on small engineering calculations.
3. Load a library and a pretrained AI model with a few lines of code.
4. Run a vision model and a language model, and judge whether their outputs can be trusted.

### Before you start
1. Click **Open in Colab** above (or open this file from the course GitHub page).
2. In Colab, choose **File → Save a copy in Drive**. Work in *your copy* so nothing is lost.
3. Run cells from top to bottom with **Shift + Enter**. Cells marked `# TODO` are yours to complete.

**Runtime:** the default CPU runtime is fine.
**Data:** `concrete_cracks_sample.zip` (photos of concrete surfaces, CC BY 4.0) — see the course `data/` folder.

**AI-use label:** **AI-Assisted** — you may ask an AI tool for explanations or debugging help. Fill in the AI-use acknowledgment at the end. You are responsible for every line.

## Part 0 — A 3-minute tour of Colab (5 min)
A notebook mixes **text cells** (like this one) and **code cells** (grey boxes with a ▶ button).
- Click a code cell and press **Shift + Enter** to run it. The result appears below the cell.
- Double-click a text cell to see how it is written; Shift + Enter to show it again.
- The **[ ]** on the left shows a number after a cell runs — that is the order cells were run.

Run the next cell.

In [ ]:
print("Hello, civil and architectural engineers!")

Python can be a calculator. Lines starting with `#` are **comments** — notes for humans that Python ignores.

In [ ]:
# A beam carries 12 kN per meter over a 6 m span. Total load in kN:
12 * 6

Now run the setup cell. Every lab starts with one: it loads libraries and a small helper, `get_data()`, that downloads course files.

In [ ]:
# Setup: run this cell first (Shift + Enter). It loads libraries and a helper to fetch course data.
import os, urllib.request
import math
import numpy as np
import matplotlib.pyplot as plt
BASE_URL = "https://raw.githubusercontent.com/TensorBuildLab/AI-Literacy-CE-ARCE/main/data/"

def get_data(name):
    """Download a course data file once and return its local path, e.g. get_data("concrete_strength.csv")."""
    if not os.path.exists(name):
        folder = os.path.dirname(name)
        if folder:
            os.makedirs(folder, exist_ok=True)
        urllib.request.urlretrieve(BASE_URL + name, name)
    return name

print("Setup complete.")

## Part 1 — Python basics with engineering numbers (25 min)

### 1.1 Variables
A **variable** is a name for a value. Use names that say what the value is, and put units in comments.

> **How Python reads your cell — four rules you will need all semester**
> - **Power is `**`**, so L² is `L**2`. (`^` means something else in Python and gives an error.)
> - **Use parentheses generously:** `(a + b) / (2 * m)` is not the same as `a + b / 2 * m`.
> - **Indentation matters:** after a line ending in `:` (`for`, `if`, `def`), indent the next lines by 4 spaces.
> - **Only the last line of a cell is shown automatically.** Use `print(...)` to show anything else.
>
> To add a new cell, hover between two cells and click **+ Code** (or **+ Text**). Keep the
> [Python cheat sheet](https://github.com/TensorBuildLab/AI-Literacy-CE-ARCE/blob/main/resources/python-cheatsheet.md) open.

In [ ]:
w = 12.0      # uniform load, kN/m
L = 6.0       # span, m

# Maximum bending moment of a simply supported beam with a uniform load: M = w·L²/8
M = w * L**2 / 8      # kN·m   (** means "to the power of")
print(M)

**f-strings** let you print numbers inside a sentence and control decimals: `{M:.1f}` means "M with 1 decimal place".

In [ ]:
print(f"Maximum moment = {M:.1f} kN·m for a {L} m span")

> 💡 **Hint:** Replace each `None` with a number or a formula. Use the same formula as above.

In [ ]:
# TODO: A longer beam: span 8 m, load 15 kN/m. Compute the maximum moment and print it with 1 decimal.
w2 = None   # kN/m
L2 = None   # m
M2 = None   # kN·m
print(f"Maximum moment = {M2} kN·m")

In [ ]:
# ✅ Self-check: run this cell to test your work above
assert M2 is not None, 'M2 is still None — finish the TODO cell above and run it (Shift + Enter).'
assert abs(M2 - 120.0) < 0.01, f'Got {M2}; expected 120.0 kN·m (15 × 8² / 8). Did you use ** for the power?'
print('Correct!')

### 1.2 Lists and loops
A **list** holds several values in order: `[3, 4.5, 6]`. A **for loop** repeats a step for each value.

In [ ]:
spans = [4.0, 5.0, 6.0, 7.0, 8.0]   # m
print("Number of spans:", len(spans))
print("First span:", spans[0], "m")     # Python counts from 0!
print("Last span:", spans[-1], "m")

for span in spans:
    moment = w * span**2 / 8
    print(f"span {span} m -> M = {moment:.1f} kN·m")

### 1.3 Decisions with `if`
Suppose the beam can safely carry a moment of 75 kN·m. Check each span.

In [ ]:
M_capacity = 75.0   # kN·m

for span in spans:
    moment = w * span**2 / 8
    if moment <= M_capacity:
        print(f"{span} m: OK      (M = {moment:.1f} ≤ {M_capacity})")
    else:
        print(f"{span} m: NOT OK  (M = {moment:.1f} > {M_capacity})")

### 1.4 Dictionaries
A **dictionary** stores labeled values: `{"name": value}`. Here is one concrete mix (kg per m³ of concrete).
The **water–cement ratio** (w/c) is one of the most important numbers in concrete: lower w/c usually means
stronger concrete.

In [ ]:
mix = {"cement": 380.0, "water": 171.0, "sand": 760.0, "gravel": 1050.0}   # kg/m³
print(mix["cement"], "kg/m³ of cement")
print("Ingredients:", list(mix.keys()))

> 💡 **Hint:** Get values out of the dictionary with `mix["water"]` and `mix["cement"]`.

In [ ]:
# TODO: compute the water-cement ratio of this mix (water divided by cement) and print it with 2 decimals.
wc = None
print(f"w/c = {wc}")

In [ ]:
# ✅ Self-check: run this cell to test your work above
assert wc is not None, 'wc is still None — finish the TODO cell above and run it.'
assert abs(wc - 0.45) < 0.01, f'Got {wc}; expected about 0.45 (171 / 380). Divide water by cement.'
print('Correct!')

### 1.5 Functions
A **function** packages a calculation so you can reuse it. `def` starts it; `return` gives back the answer.

In [ ]:
def max_moment(w, L):
    """Maximum bending moment (kN·m) of a simply supported beam with uniform load w (kN/m) and span L (m)."""
    return w * L**2 / 8

print(max_moment(12, 6))
print(max_moment(15, 8))

In [ ]:
# TODO: write a function that returns the water-cement ratio of any mix dictionary.
def wc_ratio(mix):
    """Water-cement ratio of a mix given as a dictionary with "water" and "cement" keys."""
    return None   # replace None

print(wc_ratio(mix))
print(wc_ratio({"cement": 300.0, "water": 195.0}))

In [ ]:
# ✅ Self-check: run this cell to test your work above
test = wc_ratio({'cement': 300.0, 'water': 195.0})
assert test is not None, 'Your function returns None — replace None with a formula that uses its mix argument.'
assert abs(test - 0.65) < 1e-9, f'Got {test}; expected 0.65 for water 195 / cement 300. Does your function use its own mix argument (not the variable wc from above)?'
print('Correct!')

### 1.6 Libraries
Python's power comes from **libraries** — code other people wrote. `numpy` does fast math on arrays of
numbers; `matplotlib` draws plots. We imported both in the setup cell.

In [ ]:
span_values = np.linspace(2, 10, 50)          # 50 spans from 2 m to 10 m
moments = max_moment(12, span_values)          # works on the whole array at once

plt.figure(figsize=(6, 3.5))
plt.plot(span_values, moments, label="M = wL²/8, w = 12 kN/m")
plt.axhline(M_capacity, color="red", linestyle="--", label="capacity 75 kN·m")
plt.xlabel("Span L (m)")
plt.ylabel("Max moment M (kN·m)")
plt.title("Moment grows with the square of the span")
plt.legend()
plt.show()

### 1.7 Errors are normal
Everyone gets errors. Read the **last line** of the message first — it says what went wrong. The cell below
makes a mistake on purpose (a typo in a variable name) and prints the error message instead of stopping.

In [ ]:
try:
    print(max_momnet(12, 6))      # typo on purpose
except NameError as err:
    print("Python says:", err)

**✍️ Question:** In your own words, what does that error message mean, and how would you fix it?

*Your answer:* 

## Part 2 — Activity 1: An AI model that looks at concrete (15 min)
We will use **CLIP**, a model trained by OpenAI on hundreds of millions of image–caption pairs from the
internet. It was **never trained on our crack photos**. We give it a photo and a few text descriptions, and it
picks the description that matches best. This is called **zero-shot** classification.

First, download and unzip the photos (about 7 MB).

In [ ]:
import zipfile, glob
zipfile.ZipFile(get_data("concrete_cracks_sample.zip")).extractall(".")
crack_files = sorted(glob.glob("concrete_cracks/crack/*.jpg"))
intact_files = sorted(glob.glob("concrete_cracks/no_crack/*.jpg"))
print(len(crack_files), "photos with cracks,", len(intact_files), "photos without cracks")

In [ ]:
from PIL import Image

fig, axes = plt.subplots(2, 4, figsize=(10, 5))
for ax, f in zip(axes[0], crack_files[:4]):
    ax.imshow(Image.open(f)); ax.set_title("crack"); ax.axis("off")
for ax, f in zip(axes[1], intact_files[:4]):
    ax.imshow(Image.open(f)); ax.set_title("no crack"); ax.axis("off")
plt.tight_layout(); plt.show()

Load the model. The first time, it downloads about 600 MB, which takes a minute.

In [ ]:
from transformers import pipeline
vision = pipeline("zero-shot-image-classification", model="openai/clip-vit-base-patch32")

labels = ["a photo of cracked concrete", "a photo of smooth concrete without cracks"]
result = vision(Image.open(crack_files[0]), candidate_labels=labels)
result

The model returns a **score** for each description (they add up to 1). Let's test it on 20 photos of each kind and count how often the top description is right.

In [ ]:
def classify(files, labels):
    """Return the top label for each image file."""
    return [vision(Image.open(f), candidate_labels=labels)[0]["label"] for f in files]

def accuracy(labels, n=20):
    pred_crack = classify(crack_files[:n], labels)
    pred_intact = classify(intact_files[:n], labels)
    correct = sum(p == labels[0] for p in pred_crack) + sum(p == labels[1] for p in pred_intact)
    return correct / (2 * n)

acc = accuracy(labels)
print(f"Accuracy with labels {labels}: {acc:.0%}")

Is that better than flipping a coin (50%)? The wording of the descriptions matters — a little like
wording a question to a person. Try your own.

> 💡 **Hint:** Keep the cracked description first. Try adding words like 'close-up', 'wall', or 'surface'.

In [ ]:
# TODO: write your own two descriptions (first = cracked, second = not cracked) and test them.
my_labels = ["...", "..."]
print(f"Accuracy: {accuracy(my_labels):.0%}")

**✍️ Question:** Would you let this model replace a bridge inspector? Use your accuracy numbers and at least one other reason.

*Your answer:* 

In [ ]:
# Free memory before loading the next model
del vision
import gc; gc.collect()

## Part 3 — Activity 2: A language model in Python (20 min)
Next, a **large language model (LLM)** — the same kind of model behind ChatGPT, but much smaller:
**Qwen2.5-0.5B-Instruct** has 0.5 billion parameters (big commercial models have hundreds of billions).
Small models are fast and free to run, and they make mistakes you can *see*. That is useful today.

The download is about 1 GB the first time.

In [ ]:
chat = pipeline("text-generation", model="Qwen/Qwen2.5-0.5B-Instruct")

def ask(question, max_new_tokens=250):
    """Send one question to the small language model and return its answer as text."""
    messages = [
        {"role": "system", "content": "You are a helpful assistant for civil and architectural engineering students."},
        {"role": "user", "content": question},
    ]
    out = chat(messages, max_new_tokens=max_new_tokens, do_sample=False)
    return out[0]["generated_text"][-1]["content"]

print(ask("In two sentences, what is the water-cement ratio and why does it matter for concrete?"))

**✍️ Question:** Compare the model's explanation with section 1.4 above (lower w/c usually means stronger concrete). Is the model right? Quote the sentence you agree or disagree with.

*Your answer:* 

Now a calculation you already did in Part 1. **You know the right answer** (54 kN·m). Does the model?

*Note: answers stop after a set length (`max_new_tokens`). If an answer ends mid-sentence, that is a setting, not a crash.*

In [ ]:
answer = ask("A simply supported beam spans 6 m and carries a uniform load of 12 kN/m. "
             "What is the maximum bending moment? Answer in at most 4 lines: formula, numbers, result.")
print(answer)
print("\nPython check: M = wL²/8 =", max_moment(12, 6), "kN·m")

**✍️ Question:** Was the model's answer correct? Check the **units** carefully before you decide (1 kN = 1,000 N). If something went wrong, what exactly (formula, arithmetic, units, or something else)? If it was correct, does that mean you can trust it next time?

*Your answer:* 

In [ ]:
# TODO: ask the model your own question about civil or architectural engineering.
my_question = "..."
print(ask(my_question))

**✍️ Question:** Rate the answer to your question: correct, partly correct, or wrong? How could you check it without asking another AI?

*Your answer:* 

### Optional: a much bigger model (Colab only)
Google Colab sometimes offers a built-in Gemini model to all users. If the cell below prints an answer, compare
it with the small model's answer. If it prints a message instead, that's fine — skip it.

In [ ]:
try:
    from google.colab import ai
    print(ai.generate_text("A simply supported beam spans 6 m and carries a uniform load of 12 kN/m. "
                           "What is the maximum bending moment? Show the formula and the number."))
except Exception as err:
    print("Colab's built-in model is not available here — skip this optional step.", type(err).__name__)

---
## 🎟️ Exit ticket
Three short questions you answer **alone, without AI (AI-Free)**, during the last 10 minutes of lab — even if you have not finished every part above. Use your own words, 1–3 sentences each. (If you must leave early, answer them on your own before you submit.)

**1.** What is the difference between a code cell and a text cell?

*Your answer:* 

**2.** The CLIP model never saw our crack photos during training. How could it still sort them — and what is one reason it sometimes fails?

*Your answer:* 

**3.** Name one thing the language model got wrong or vague today, and one way you could check an AI answer like that.

*Your answer:* 

---
## 🤖 AI-use acknowledgment
Did you use an AI tool (ChatGPT, Gemini, Copilot, Claude, Colab's AI features, etc.) while working on this notebook? (The small model that this notebook's own code runs does not count.)
- If **no**, write: *"I did not use AI tools for this assignment."*
- If **yes**, write: *"I acknowledge the use of [tool and version] to [specific use]. Prompts I used included [paste or summarize]. I checked the output by [how I verified it]. I am responsible for this work."*

Same template as the course [AI-use guide](https://github.com/TensorBuildLab/AI-Literacy-CE-ARCE/blob/main/resources/ai-use-guide.md).

*Your statement:* 

## 📤 How to submit
**Due:** Sunday 11:59 pm (Arizona time) of this week, on Brightspace. Late work: see the syllabus late-work table (two grace days per semester, then −10% per day, up to 3 days).

1. **Runtime → Restart session and run all** — make sure everything runs without errors (cells that load a model take a few extra minutes).
2. **File → Download → Download .ipynb**.
3. Upload the `.ipynb` file to the matching assignment on **Brightspace**.